# AI Healthcare NLP Symptom Checker
Module 03 project. Classroom-only, fictional inputs.

Run the cells top to bottom. The **Symptom Recognition** tab is not diagnostic; the **Fictional Triage** tab is an if/elif teaching example, not patient-facing triage.

## Research: Natural Language Processing in Healthcare

### Introduction

Natural Language Processing (NLP) is a technology that helps computers understand and process human language. In healthcare, NLP can help identify symptoms from information that patients or doctors write. For my project, I created a simple symptom checker that uses NLP techniques to recognize symptoms from sentences entered by users.

### Why I Chose This Project

I chose this project because I wanted to learn how AI and NLP can be used in healthcare. People describe their symptoms in different ways, and NLP can help computers recognize important information from those descriptions.

Research shows that NLP has been used to identify symptoms from electronic health records. One systematic review studied 27 research papers about using NLP to extract symptom information. This shows how useful NLP can be in healthcare.

### How My Application Works

My application allows users to enter a sentence describing their symptoms. It then processes the text and looks for symptoms using a small database.

First, the application converts the text to lowercase and processes some common negative expressions. Next, it searches for symptoms and their different word forms. For example, it can recognize both "headache" and "headaches."

The application also tries to understand negative statements, such as "I do not have a fever." It separates symptoms that are reported from those that are denied or unclear.

After analyzing the text, the application displays the recognized symptoms and some general health information. I also added a separate section that uses fictional examples to demonstrate how a rule-based system can assign preliminary assessments and triage levels.

### Limitations

Although my application can recognize several common symptoms, it has some limitations. It uses a small symptom database and simple rules, so it may not understand complicated sentences or recognize every symptom.

It also does not use a trained machine learning model and cannot provide an accurate medical diagnosis. The application is mainly designed to demonstrate how NLP works.

### Ethical Considerations

Privacy and accuracy are very important when using AI in healthcare. Incorrect information could affect a patient's health, and personal medical information needs to be protected.

For this reason, my application is designed for educational purposes only. It does not intentionally save patient information, and users should not enter real personal medical details. I also added a disclaimer explaining that the application cannot replace professional medical advice.

The World Health Organization also emphasizes the importance of safety, fairness, privacy, and transparency when developing AI systems for healthcare.

### Conclusion

Through this project, I learned how NLP can be used to recognize symptoms from text and how simple Python rules can process information. I also learned that developing AI applications for healthcare requires more than just writing code. Accuracy, privacy, and patient safety are equally important.

Although my application is simple, it helped me understand how NLP can be applied to real-world healthcare problems.

### References

1. *Natural language processing of symptoms documented in free-text narratives of electronic health records: A systematic review.* (2019). Journal of the American Medical Informatics Association.  
https://pmc.ncbi.nlm.nih.gov/articles/PMC6657282/

2. *Using natural language processing to extract information from clinical text in electronic medical records for populating clinical registries: A systematic review.* (2025).  
https://pubmed.ncbi.nlm.nih.gov/41093296/

3. World Health Organization. (2021). *Ethics and governance of artificial intelligence for health.*  
https://www.who.int/publications/i/item/9789240029200

In [1]:
!pip -q install "gradio>=5,<7"

## Core logic and interface

In [2]:
"""Educational NLP healthcare project. Not for real patient care."""
import re
import gradio as gr

SYMPTOMS = {
    'headache': ['headache', 'headaches'],
    'fever': ['fever', 'fevers'],
    'cough': ['cough', 'coughing', 'coughs'],
    'fatigue': ['fatigue', 'tiredness', 'tired'],
    'nausea': ['nausea', 'nauseous'],
    'dizziness': ['dizziness', 'dizzy'],
    'sore throat': ['sore throat'],
    'stomach pain': ['stomach pain', 'stomachache'],
    'chest pain': ['chest pain'],
    'breathing difficulty': ['difficulty breathing', 'shortness of breath'],
}

INFO = {
    'headache': 'Headaches can have many causes.',
    'fever': 'Fever can be associated with illness.',
    'cough': 'Cough is a common respiratory symptom.',
    'fatigue': 'Fatigue describes unusual tiredness.',
    'nausea': 'Nausea means feeling sick to your stomach.',
    'dizziness': 'Dizziness can describe imbalance or lightheadedness.',
    'sore throat': 'Sore throat describes discomfort in the throat.',
    'stomach pain': 'Abdominal discomfort has many possible causes.',
    'chest pain': 'Chest pain can have serious causes.',
    'breathing difficulty': 'Breathing difficulty can have serious causes.',
}

# This is only a lesson in if/elif logic on FIXED FICTIONAL cases.
# It is not a classifier for user-entered symptoms.
FICTIONAL_CASES = {
    'Case A: mild fictional cold-like symptoms': {
        'assessment': 'Illustrative possibility: a common cold (not a diagnosis).',
        'triage': 'PRIMARY CARE / SELF-CARE (fictional teaching label)',
        'why': 'A teaching example of a lower-acuity rule; symptoms alone do not establish safety.'
    },
    'Case B: fictional worsening symptoms': {
        'assessment': 'Illustrative possibility: an illness requiring clinical assessment (not a diagnosis).',
        'triage': 'URGENT (fictional teaching label)',
        'why': 'A teaching example of a rule that escalates when a scenario is described as worsening.'
    },
    'Case C: fictional severe chest pain and difficulty breathing': {
        'assessment': 'Illustrative possibility: a serious emergency (not a diagnosis).',
        'triage': 'EMERGENCY (fictional teaching label)',
        'why': 'A teaching example showing that severe warning signs should not be dismissed.'
    },
}

DISCLAIMER = ('EDUCATIONAL DEMONSTRATION ONLY. No diagnosis or real-world triage is provided. '
              'This small rule-based system may misread negation and miss symptoms. '
              'Do not enter real patient or identifiable health data. '
              'For health concerns contact a healthcare professional; if you think '
              'there is a medical emergency, contact local emergency services.')


def _normalize(text):
    """Normalize common negation contractions before matching."""
    text = text.lower().replace('’', "'")
    for a, b in {
        "don't": 'do not', "doesn't": 'does not', "didn't": 'did not',
        "isn't": 'is not', "haven't": 'have not', "hasn't": 'has not',
        "can't": 'cannot', "won't": 'will not',
    }.items():
        text = re.sub(r'\b' + re.escape(a) + r'\b', b, text)
    return text


def extract_symptoms(text):
    """Return present, denied, and uncertain symptom sets from basic rules.

    The rules cannot reliably interpret complex natural language.
    """
    if not text or not text.strip():
        return [], [], []
    text = _normalize(text)
    # Separate independent statements; don't split a list of symptoms at 'and'.
    clauses = re.split(r'\bbut\b|[.;!?]', text)
    present, denied, uncertain = set(), set(), set()
    for clause in clauses:
        matches = []
        for name, aliases in SYMPTOMS.items():
            for alias in aliases:
                match = re.search(r'\b' + re.escape(alias) + r'\b', clause)
                if match:
                    matches.append((match.start(), match.end(), name))
                    break
        matches.sort()
        for start, _, name in matches:
            preceding = clause[max(0, start - 45):start]
            # Recognize only a small subset of negative expressions.
            negative = re.search(r'\b(?:no|not|without|never|cannot)\b(?:\W+\w+){0,4}\W*$', preceding)
            # Uncertainty isn't the same as affirming a symptom.
            unsure = re.search(r'\b(?:maybe|possibly|unsure|might)\b', clause)
            if unsure:
                uncertain.add(name)
            elif negative:
                denied.add(name)
            else:
                present.add(name)
    # Contradictory mentions should be flagged for review, not resolved blindly.
    conflicted = present & denied
    present -= conflicted
    denied -= conflicted
    uncertain |= conflicted
    return sorted(present), sorted(denied), sorted(uncertain)


def analyze_symptoms(text):
    """Create a *non-diagnostic* educational summary of symptom mentions."""
    if not text or not text.strip():
        return 'Enter a fictional example sentence.\n\n' + DISCLAIMER
    present, denied, uncertain = extract_symptoms(text)
    lines = ['SYMPTOM RECOGNITION (NOT A DIAGNOSIS)', '']
    lines += ['Reported: ' + (', '.join(present) or 'none recognized')]
    lines += ['Denied: ' + (', '.join(denied) or 'none recognized')]
    lines += ['Uncertain/conflicting: ' + (', '.join(uncertain) or 'none recognized')]
    lines += ['', 'GENERAL EDUCATIONAL INFORMATION:']
    lines += [f'- {name.title()}: {INFO[name]}' for name in present]
    if not present:
        lines += ['- No confirmed symptom matches to describe.']
    lines += ['', 'Preliminary diagnosis: NOT DETERMINED from free text.',
              'Personal triage level: NOT DETERMINED from free text.', '', DISCLAIMER]
    return '\n'.join(lines)


def demonstrate_triage(case):
    """Illustrate preliminary-assessment + triage labels for fictional cases."""
    if case not in FICTIONAL_CASES:
        return 'Choose one of the supplied fictional cases.\n\n' + DISCLAIMER
    # Explicit if/elif statements meet the coursework's rule-based logic aim.
    if case.startswith('Case C'):
        record = FICTIONAL_CASES[case]
    elif case.startswith('Case B'):
        record = FICTIONAL_CASES[case]
    else:
        record = FICTIONAL_CASES[case]
    return (f"FICTIONAL SCENARIO: {case}\n\n"
            f"Preliminary assessment (illustrative): {record['assessment']}\n"
            f"Example triage label: {record['triage']}\n"
            f"Why the rule selected it: {record['why']}\n\n{DISCLAIMER}")


def create_app():
    """Create a Gradio interface with separate NLP and teaching-demo tabs."""
    with gr.Blocks(title='AI Healthcare NLP Learning Demo') as demo:
        gr.Markdown('# AI Healthcare: Symptom Recognition & Triage Logic')
        gr.Markdown('**Classroom project only. Use fictional examples, never patient data.**')
        with gr.Tab('Symptom recognition'):
            text = gr.Textbox(label='Fictional symptom statement', lines=3,
                placeholder='I have coughing and nausea, but no headache.')
            output = gr.Textbox(label='NLP results', lines=17)
            gr.Button('Analyze text').click(analyze_symptoms, inputs=text, outputs=output)
            gr.Examples(examples=[['I have a headache and fever.'],
                ['I have coughing and nausea, but no headache.'],
                ["I don't have fever, but I feel dizzy."],
                ['I might have a headache.']], inputs=text)
        with gr.Tab('Fictional triage example'):
            gr.Markdown('These labels come from preset **fictional cases**, not user-entered symptoms.')
            case = gr.Dropdown(choices=list(FICTIONAL_CASES),
                value=list(FICTIONAL_CASES)[0], label='Select a fictional scenario')
            result = gr.Textbox(label='Teaching example', lines=12)
            gr.Button('Show rule-based assessment').click(demonstrate_triage,
                inputs=case, outputs=result)
        gr.Markdown('**Privacy:** This demo does not intentionally save entered text. '
                    'Colab/Gradio hosting services may still process content. '
                    'Do not enter real medical information. No public sharing link is created by default.')
    return demo





## Open the interface
This launches inside Colab. Do not enter real patient data.

In [3]:
demo = create_app()
demo.launch(share=False, inline=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

## Automated functionality tests
These check program logic, not medical accuracy.

In [4]:
"""Tests for educational functionality; not medical validity tests."""
import unittest


class Tests(unittest.TestCase):
    def test_basic(self):
        self.assertEqual(extract_symptoms('I have a headache and fever.'), (['fever', 'headache'], [], []))
    def test_negation(self):
        self.assertEqual(extract_symptoms('I have coughing and nausea, but no headache.'), (['cough', 'nausea'], ['headache'], []))
    def test_contractions(self):
        self.assertEqual(extract_symptoms("I don't have fever, but I feel dizzy."), (['dizziness'], ['fever'], []))
    def test_multiword(self):
        self.assertEqual(extract_symptoms('I have a sore throat and fatigue.'), (['fatigue', 'sore throat'], [], []))
    def test_uncertainty(self):
        self.assertEqual(extract_symptoms('I might have a headache.'), ([], [], ['headache']))
    def test_word_boundary(self):
        self.assertEqual(extract_symptoms('I read a headachebook.'), ([], [], []))
    def test_no_input(self):
        self.assertIn('Enter a fictional', analyze_symptoms(''))
    def test_not_diagnosing(self):
        self.assertIn('NOT DETERMINED', analyze_symptoms('I have fever'))
    def test_fictional_triage(self):
        self.assertIn('EMERGENCY (fictional teaching label)', demonstrate_triage(list(FICTIONAL_CASES)[2]))

suite = unittest.defaultTestLoader.loadTestsFromTestCase(Tests)
result = unittest.TextTestRunner(verbosity=2).run(suite)


test_basic (__main__.Tests.test_basic) ... ok
test_contractions (__main__.Tests.test_contractions) ... ok
test_fictional_triage (__main__.Tests.test_fictional_triage) ... ok
test_multiword (__main__.Tests.test_multiword) ... ok
test_negation (__main__.Tests.test_negation) ... ok
test_no_input (__main__.Tests.test_no_input) ... ok
test_not_diagnosing (__main__.Tests.test_not_diagnosing) ... ok
test_uncertainty (__main__.Tests.test_uncertainty) ... ok
test_word_boundary (__main__.Tests.test_word_boundary) ... ok

----------------------------------------------------------------------
Ran 9 tests in 0.036s

OK
